<div style="
background:linear-gradient(135deg,#0B1720 0%,#132832 48%,#0A151D 100%);
border:2px solid rgba(214,184,112,0.42);
border-radius:22px;
padding:30px 34px;
font-family:'Segoe UI',Arial,sans-serif;
box-shadow:0 15px 45px rgba(0,0,0,0.28);
">

<div style="
font-size:13px;
letter-spacing:4px;
text-transform:uppercase;
color:#D6B870;
font-weight:700;
margin-bottom:14px;
">
HANDS-ON LAB
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:30px;
color:#F7F3E9;
margin-bottom:18px;
">
Tuned End-to-End Pipeline
</div>

<p style="
font-size:17px;
line-height:1.8;
color:#B7C5C8;
margin:0;
">
In this hands-on lab, we build a complete machine learning pipeline,
tune its hyperparameters using cross-validation, and evaluate the
final model on unseen test data.
</p>

<div style="
margin-top:20px;
padding:15px 18px;
border-radius:12px;
background:rgba(141,178,167,0.06);
border:1px solid rgba(141,178,167,0.20);
font-family:'Courier New',monospace;
font-size:16px;
color:#A9C4BA;
">
Preprocessing → Feature Engineering → Tuning → Evaluation
</div>

</div>

<div style="
background:linear-gradient(135deg,#0B1720,#132832,#0A151D);
border-left:4px solid #D6B870;
border-radius:16px;
padding:22px 26px;
font-family:'Segoe UI',Arial,sans-serif;
">

<div style="color:#D6B870;font-size:12px;letter-spacing:3px;font-weight:700;">
STEP 01
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:26px;
color:#F7F3E9;
margin:8px 0 10px;
">
Build the Preprocessing Pipeline
</div>

<p style="color:#B7C5C8;font-size:16px;line-height:1.7;margin:0;">
Build a <strong style="color:#E2CC94;">ColumnTransformer</strong>
to preprocess numerical and categorical features, then combine it
with the machine learning model inside a single Pipeline.
</p>

</div>

In [28]:
import pandas as pd
import numpy as np

from IPython.display import HTML, display

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [29]:
import pandas as pd

df = pd.read_csv("hotel_bookings.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [30]:
X = df.drop("is_canceled", axis=1)
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
numeric_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Numeric Columns:")
print(numeric_cols)

print("\nCategorical Columns:")
print(categorical_cols)

Numeric Columns:
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

Categorical Columns:
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'reservation_status', 'reservation_status_date']


In [32]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

<div style="
background:linear-gradient(135deg,#0B1720,#132832,#0A151D);
border-left:4px solid #D6B870;
border-radius:16px;
padding:22px 26px;
font-family:'Segoe UI',Arial,sans-serif;
">

<div style="color:#D6B870;font-size:12px;letter-spacing:3px;font-weight:700;">
STEP 02
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:26px;
color:#F7F3E9;
margin:8px 0 10px;
">
Add Engineered Features
</div>

<p style="color:#B7C5C8;font-size:16px;line-height:1.7;margin:0;">
The engineered features created during Day 4 are included in the
training data before the pipeline is tuned.
</p>

</div>

In [33]:
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Features:", X_train.columns.tolist())

Training shape: (95512, 31)
Test shape: (23878, 31)
Features: ['hotel', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']


<div style="
background:linear-gradient(135deg,#0B1720 0%,#132832 48%,#0A151D 100%);
border:2px solid rgba(214,184,112,0.42);
border-radius:22px;
padding:30px 34px;
font-family:'Segoe UI',Arial,sans-serif;
box-shadow:0 15px 45px rgba(0,0,0,0.28);
">

<div style="
font-size:13px;
letter-spacing:4px;
text-transform:uppercase;
color:#D6B870;
font-weight:700;
margin-bottom:14px;
">
STEP 03
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:30px;
color:#F7F3E9;
margin-bottom:18px;
">
Define a Model and Its Hyperparameter Grid
</div>

<p style="
font-size:17px;
line-height:1.8;
color:#B7C5C8;
margin:0;
">
A Random Forest Classifier is selected as the model, and a
hyperparameter grid is defined to test different configurations
during GridSearchCV.
</p>

</div>


In [34]:
model = RandomForestClassifier(
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

param_grid = {
    "model__n_estimators": [100],
    "model__max_depth": [10, 20],
    "model__min_samples_split": [2]
}

<div style="
background:linear-gradient(135deg,#0B1720 0%,#132832 48%,#0A151D 100%);
border:2px solid rgba(214,184,112,0.42);
border-radius:22px;
padding:30px 34px;
font-family:'Segoe UI',Arial,sans-serif;
box-shadow:0 15px 45px rgba(0,0,0,0.28);
">

<div style="
font-size:13px;
letter-spacing:4px;
text-transform:uppercase;
color:#D6B870;
font-weight:700;
margin-bottom:14px;
">
STEP 04
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:30px;
color:#F7F3E9;
margin-bottom:18px;
">
Run GridSearchCV
</div>

<p style="
font-size:17px;
line-height:1.8;
color:#B7C5C8;
margin:0;
">
GridSearchCV is configured to evaluate all hyperparameter
combinations using <strong style="color:#D6CC94;">5-fold cross-validation</strong>
and <strong style="color:#D6CC94;">F1 Score</strong> to identify the
best-performing configuration.
</p>

<div style="
margin-top:20px;
padding:15px 18px;
border-radius:12px;
background:rgba(141,178,167,0.06);
border:1px solid rgba(141,178,167,0.20);
font-family:'Courier New',monospace;
font-size:16px;
color:#A9C4BA;
">
Pipeline → GridSearchCV → CV = 5 → F1
</div>

</div>

In [35]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1"
)

print("GridSearchCV configured successfully.")

GridSearchCV configured successfully.


In [36]:
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [10, 20], 'model__min_samples_split': [2], 'model__n_estimators': [100]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Con

<div style="
background:linear-gradient(135deg,#0B1720 0%,#132832 48%,#0A151D 100%);
border:2px solid rgba(214,184,112,0.42);
border-radius:22px;
padding:30px 34px;
font-family:'Segoe UI',Arial,sans-serif;
box-shadow:0 15px 45px rgba(0,0,0,0.28);
">

<div style="
font-size:13px;
letter-spacing:4px;
text-transform:uppercase;
color:#D6B870;
font-weight:700;
margin-bottom:14px;
">
STEP 05
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:30px;
color:#F7F3E9;
margin-bottom:18px;
">
Report the Best Hyperparameters and Best CV Score
</div>

<p style="
font-size:17px;
line-height:1.8;
color:#B7C5C8;
margin:0;
">
The best hyperparameter combination and the highest mean
<strong style="color:#E2CC94;">F1 Score</strong> obtained during
5-fold cross-validation are extracted from GridSearchCV.
</p>

<div style="
margin-top:20px;
padding:15px 18px;
border-radius:12px;
background:rgba(141,178,167,0.06);
border:1px solid rgba(141,178,167,0.20);
font-family:'Courier New',monospace;
font-size:16px;
color:#A9C4BA;
">
best_params_ → Optimal Configuration<br>
best_score_ → Best Mean CV F1 Score
</div>

</div>

In [37]:
print("Best Hyperparameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(grid_search.best_score_)

Best Hyperparameters:
{'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 100}

Best Cross-Validation F1 Score:
1.0


<div style="
background:linear-gradient(135deg,#0B1720 0%,#132832 48%,#0A151D 100%);
border:2px solid rgba(214,184,112,0.42);
border-radius:22px;
padding:30px 34px;
font-family:'Segoe UI',Arial,sans-serif;
box-shadow:0 15px 45px rgba(0,0,0,0.28);
">

<div style="
font-size:13px;
letter-spacing:4px;
text-transform:uppercase;
color:#D6B870;
font-weight:700;
margin-bottom:14px;
">
STEP 06
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:30px;
color:#F7F3E9;
margin-bottom:18px;
">
Evaluate the Final Tuned Pipeline
</div>

<p style="
font-size:17px;
line-height:1.8;
color:#B7C5C8;
margin:0;
">
The best pipeline selected by GridSearchCV is evaluated on the
<strong style="color:#E2CC94;">held-out test set</strong> to measure
its performance on unseen data.
</p>

<div style="
margin-top:20px;
padding:15px 18px;
border-radius:12px;
background:rgba(141,178,167,0.06);
border:1px solid rgba(141,178,167,0.20);
font-family:'Courier New',monospace;
font-size:16px;
color:#A9C4BA;
">
Best Pipeline → X_test → Predictions → Evaluation
</div>

</div>

In [38]:
best_pipeline = grid_search.best_estimator_

y_pred = best_pipeline.predict(X_test)

print("Test F1 Score:", f1_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Test F1 Score: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     15033
           1       1.00      1.00      1.00      8845

    accuracy                           1.00     23878
   macro avg       1.00      1.00      1.00     23878
weighted avg       1.00      1.00      1.00     23878



<div style="
background:linear-gradient(135deg,#0B1720 0%,#132832 48%,#0A151D 100%);
border:2px solid rgba(214,184,112,0.42);
border-radius:22px;
padding:30px 34px;
font-family:'Segoe UI',Arial,sans-serif;
box-shadow:0 15px 45px rgba(0,0,0,0.28);
">

<div style="
font-size:13px;
letter-spacing:4px;
text-transform:uppercase;
color:#D6B870;
font-weight:700;
margin-bottom:14px;
">
STEP 07
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:30px;
color:#F7F3E9;
margin-bottom:18px;
">
Compare the Tuned Pipeline Against the Baseline Model
</div>

<p style="
font-size:17px;
line-height:1.8;
color:#B7C5C8;
margin:0;
">
The tuned pipeline is compared with the baseline model using the same
held-out test set and <strong style="color:#E2CC94;">F1 Score</strong>
to determine whether hyperparameter tuning improved performance.
</p>

<div style="
margin-top:20px;
padding:15px 18px;
border-radius:12px;
background:rgba(141,178,167,0.06);
border:1px solid rgba(141,178,167,0.20);
font-family:'Courier New',monospace;
font-size:16px;
color:#A9C4BA;
">
Baseline F1 → Tuned F1 → Performance Comparison
</div>

</div>

In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

baseline_f1 = f1_score(y_test, baseline_pred)

print("Baseline F1 Score:", baseline_f1)

Baseline F1 Score: 1.0


In [40]:
baseline_pred = baseline_model.predict(X_test)

baseline_f1 = f1_score(y_test, baseline_pred)
tuned_f1 = f1_score(y_test, y_pred)

print("Baseline F1 Score:", baseline_f1)
print("Tuned F1 Score:", tuned_f1)
print("Improvement:", tuned_f1 - baseline_f1)

Baseline F1 Score: 1.0
Tuned F1 Score: 1.0
Improvement: 0.0


<div style="
background:linear-gradient(135deg,#0B1720 0%,#132832 48%,#0A151D 100%);
border:2px solid rgba(214,184,112,0.42);
border-radius:22px;
padding:30px 34px;
font-family:'Segoe UI',Arial,sans-serif;
box-shadow:0 15px 45px rgba(0,0,0,0.28);
">

<div style="
font-size:13px;
letter-spacing:4px;
text-transform:uppercase;
color:#D6B870;
font-weight:700;
margin-bottom:14px;
">
STEP 08
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:30px;
color:#F7F3E9;
margin-bottom:18px;
">
Confirm No Data Leakage
</div>

<p style="
font-size:17px;
line-height:1.8;
color:#B7C5C8;
margin:0;
">
All preprocessing steps are contained inside the
<strong style="color:#E2CC94;">Pipeline</strong>.
The <strong style="color:#E2CC94;">ColumnTransformer</strong> performs
imputation, scaling, and encoding only during the appropriate training
steps of the pipeline, preventing information from the test set from
leaking into model training.
</p>

<div style="
margin-top:20px;
padding:15px 18px;
border-radius:12px;
background:rgba(141,178,167,0.06);
border:1px solid rgba(141,178,167,0.20);
font-family:'Courier New',monospace;
font-size:16px;
color:#A9C4BA;
">
X_train → Pipeline → Preprocessing → Model<br>
X_test → Pipeline → Prediction
</div>

</div>

In [41]:
print("Pipeline steps:")
print(best_pipeline.named_steps)

print("\nPreprocessing is inside the Pipeline:", "preprocessor" in best_pipeline.named_steps)

Pipeline steps:
{'preprocessor': ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['lead_time', 'arrival_date_year',
                                  'arrival_date_week_number',
                                  'arrival_date_day_of_month',
                                  'stays_in_weekend_nights',
                                  'stays_in_week_nights', 'adults', 'children',
                                  'babies', 'is_repeated_guest',
                                  'previous_cancellations',
                                  'previous...
                                  'total_of_special_requests']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                            

<div style="
background:linear-gradient(135deg,#0B1720 0%,#132832 48%,#0A151D 100%);
border:2px solid rgba(214,184,112,0.42);
border-radius:22px;
padding:30px 34px;
font-family:'Segoe UI',Arial,sans-serif;
box-shadow:0 15px 45px rgba(0,0,0,0.28);
">

<div style="
font-size:13px;
letter-spacing:4px;
text-transform:uppercase;
color:#D6B870;
font-weight:700;
margin-bottom:14px;
">
STEP 09
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:30px;
color:#F7F3E9;
margin-bottom:18px;
">
Final Results & Tuning Analysis
</div>

<p style="
font-size:17px;
line-height:1.8;
color:#B7C5C8;
margin:0 0 18px 0;
">
The tuned Random Forest pipeline was evaluated using 5-fold cross-validation
and then tested on the held-out test set.
</p>

<div style="
display:grid;
grid-template-columns:1fr 1fr;
gap:12px;
">

<div style="
padding:15px 18px;
border-radius:12px;
background:rgba(214,184,112,0.06);
border:1px solid rgba(214,184,112,0.18);
color:#E2CC94;
font-family:'Courier New',monospace;
">
Best CV F1 → 1.0
</div>

<div style="
padding:15px 18px;
border-radius:12px;
background:rgba(141,178,167,0.06);
border:1px solid rgba(141,178,167,0.18);
color:#A9C4BA;
font-family:'Courier New',monospace;
">
Test F1 → 1.0
</div>

<div style="
padding:15px 18px;
border-radius:12px;
background:rgba(214,184,112,0.06);
border:1px solid rgba(214,184,112,0.18);
color:#E2CC94;
font-family:'Courier New',monospace;
">
Baseline F1 → 1.0
</div>

<div style="
padding:15px 18px;
border-radius:12px;
background:rgba(141,178,167,0.06);
border:1px solid rgba(141,178,167,0.18);
color:#A9C4BA;
font-family:'Courier New',monospace;
">
Improvement → 0.0
</div>

</div>

<p style="
margin-top:20px;
font-size:16px;
line-height:1.8;
color:#B7C5C8;
">
Hyperparameter tuning did not improve the F1 Score because the baseline
model already achieved a perfect F1 Score of 1.0. The tuned pipeline
maintained the same performance while providing an optimized set of
hyperparameters.
</p>

</div>

In [42]:
from IPython.display import HTML, display

print("Final Results")
print("-" * 40)
print("Best Parameters:", grid_search.best_params_)
print("Best CV F1 Score:", grid_search.best_score_)
print("Baseline Test F1:", baseline_f1)
print("Tuned Test F1:", tuned_f1)
print("Improvement:", tuned_f1 - baseline_f1)

display(HTML("""
<div style="
background:linear-gradient(135deg,#0B1720 0%,#132832 48%,#0A151D 100%);
border:2px solid rgba(214,184,112,0.42);
border-radius:22px;
padding:30px 34px;
font-family:'Segoe UI',Arial,sans-serif;
box-shadow:0 15px 45px rgba(0,0,0,0.28);
">

<div style="
font-size:13px;
letter-spacing:4px;
text-transform:uppercase;
color:#D6B870;
font-weight:700;
margin-bottom:14px;
">
FINAL RESULTS
</div>

<div style="
font-family:Georgia,'Times New Roman',serif;
font-size:30px;
color:#F7F3E9;
margin-bottom:22px;
">
Tuned Pipeline Performance
</div>

<div style="
display:grid;
grid-template-columns:1fr 1fr;
gap:14px;
">

<div style="
padding:17px 20px;
border-radius:14px;
background:rgba(214,184,112,0.06);
border:1px solid rgba(214,184,112,0.20);
">
<div style="font-size:12px;color:#8FA4AA;letter-spacing:1px;margin-bottom:7px;">
BEST CV F1 SCORE
</div>
<div style="font-size:25px;color:#E2CC94;font-weight:700;">
1.0
</div>
</div>

<div style="
padding:17px 20px;
border-radius:14px;
background:rgba(141,178,167,0.06);
border:1px solid rgba(141,178,167,0.20);
">
<div style="font-size:12px;color:#8FA4AA;letter-spacing:1px;margin-bottom:7px;">
TUNED TEST F1
</div>
<div style="font-size:25px;color:#A9C4BA;font-weight:700;">
1.0
</div>
</div>

<div style="
padding:17px 20px;
border-radius:14px;
background:rgba(141,178,167,0.06);
border:1px solid rgba(141,178,167,0.20);
">
<div style="font-size:12px;color:#8FA4AA;letter-spacing:1px;margin-bottom:7px;">
BASELINE TEST F1
</div>
<div style="font-size:25px;color:#A9C4BA;font-weight:700;">
1.0
</div>
</div>

<div style="
padding:17px 20px;
border-radius:14px;
background:rgba(214,184,112,0.06);
border:1px solid rgba(214,184,112,0.20);
">
<div style="font-size:12px;color:#8FA4AA;letter-spacing:1px;margin-bottom:7px;">
IMPROVEMENT
</div>
<div style="font-size:25px;color:#E2CC94;font-weight:700;">
0.0
</div>
</div>

</div>

<div style="
margin-top:22px;
padding:18px 20px;
border-radius:14px;
background:rgba(255,255,255,0.025);
border:1px solid rgba(255,255,255,0.08);
">

<div style="
font-size:12px;
letter-spacing:2px;
text-transform:uppercase;
color:#D6B870;
font-weight:700;
margin-bottom:9px;
">
Best Hyperparameters
</div>

<div style="
font-family:'Courier New',monospace;
font-size:15px;
line-height:1.8;
color:#B7C5C8;
">
Random Forest → max_depth = 20<br>
Random State → 42
</div>

</div>

<div style="
margin-top:20px;
padding-top:18px;
border-top:1px solid rgba(255,255,255,0.08);
font-size:16px;
line-height:1.7;
color:#8FA4AA;
">
Tuning maintained the baseline performance. Since the baseline already
achieved an F1 Score of 1.0, there was no measurable improvement from
hyperparameter tuning.
</div>

</div>
"""))

Final Results
----------------------------------------
Best Parameters: {'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Best CV F1 Score: 1.0
Baseline Test F1: 1.0
Tuned Test F1: 1.0
Improvement: 0.0
